In [ ]:
# 4. Evaluation Metrics for Classification

### Decision Threshold in Machine Learning

The decision threshold in machine learning is a crucial factor in classifying data into binary classes. 

It is the value that determines how likely an instance is to be classified as a positive or negative class based on the model's output probabilities. 

For example, in binary classification, a model may output a probability score between 0 and 1, and if the score exceeds the threshold, the instance is classified as positive; otherwise, it is classified as negative. 

The default threshold is often set to 0.5, but this value may not be optimal for all scenarios, especially in cases with imbalanced classes or where the cost of false negatives outweighs the cost of false positives.

### Linspace in Python

The linspace function in Python's NumPy library generates evenly spaced numbers over a specified interval. 

It is commonly used for creating arrays with a specified number of elements between two endpoints.

## 4.1 Evaluation Metrics: Session Overview

**Metric** is a function that compares the predictions with the actual values and outputs a single number that tells how good the predictions are.

Inputs are predictions and true values. Outputs are a single number. 

Diffferent metrics look at different aspects of the predections. Picking the right metrick for the problemm is the topic of this entire module.

### Environment

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# SciKit Learn

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

### Data Read and Preparation

In [5]:
df = pd.read_csv('../03-classification/data-week-3.csv')

In [11]:
# Take spaces out of column headers
df.columns = df.columns.str.lower().str.replace(' ', '_')

# the below is 'object' is the instructor's repo; changed to 'str' here
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)

# Everything everywhwere all at once
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

# Unparseable strings will be replaced with NaN
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

# Make values in churn column 0 or 1 instead of no or yes
df.churn = (df.churn == 'yes').astype(int)

In [12]:
df.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-vhveg,female,0,yes,no,1,no,no_phone_service,dsl,no,...,no,no,no,no,month-to-month,yes,electronic_check,29.85,29.85,0
1,5575-gnvde,male,0,no,no,34,yes,no,dsl,yes,...,yes,no,no,no,one_year,no,mailed_check,56.95,1889.50,0
2,3668-qpybk,male,0,no,no,2,yes,no,dsl,yes,...,no,no,no,no,month-to-month,yes,mailed_check,53.85,108.15,0
3,7795-cfocw,male,0,no,no,45,no,no_phone_service,dsl,yes,...,yes,yes,no,no,one_year,no,bank_transfer_(automatic),42.30,1840.75,0
4,9237-hqitu,female,0,no,no,2,yes,no,fiber_optic,no,...,no,no,no,no,month-to-month,yes,electronic_check,70.70,151.65,0


### Split the data into training, validation and test groups

Set up the **validation framework**.

Then delete the target 'churn' column

In [13]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

del df_train['churn']
del df_val['churn']
del df_test['churn']

In [14]:
df_val.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges
0,5846-neqvz,male,0,yes,yes,71,yes,no,dsl,yes,no,yes,yes,no,yes,two_year,yes,credit_card_(automatic),70.85,4973.40
1,3645-deygf,male,0,no,yes,1,yes,no,no,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,month-to-month,no,mailed_check,20.75,20.75
2,3590-tcxtb,male,0,no,no,1,yes,no,no,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,month-to-month,no,mailed_check,20.35,20.35
3,8433-wxgna,male,0,no,no,2,yes,no,fiber_optic,yes,no,no,no,no,no,month-to-month,yes,electronic_check,75.70,189.20
4,2654-vbvpb,female,0,no,no,1,yes,no,no,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,month-to-month,no,bank_transfer_(automatic),19.90,19.90


### Define Numerical and Categorical Values

In [15]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

### [[[HHHHHHHH]]]

Heavy lifting happens here

**Feature importance* and **one-hot encoding** that featured in the last lesson are skipped at this point.

`Dictvectorizer()` is a SciKitLearn transformer. It has the `fit` step and the `transform` step.

In [18]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

In [19]:
model = LogisticRegression(solver='lbfgs', max_iter=1000)
model.fit(X_train, y_train)

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)

## 4.4 Precision and Recall

## 4.5 ROC Curves

### Random Model

### Ideal Model